# CoinSight — OHLCV exploratory data analysis

Notebook đọc trực tiếp từ Data Warehouse để:

1. kiểm tra coverage và chất lượng dữ liệu của **toàn bộ coin**;
2. xem trend, candlestick và moving average theo ngày hoặc giờ;
3. đo correlation giữa các coin, trong từng nhóm và giữa toàn bộ nhóm coin.

> Correlation được tính trên **log-return**, không tính trực tiếp trên mức giá. Hai chuỗi giá cùng tăng theo thời gian có thể tạo correlation giả dù biến động từng kỳ không liên quan.

## 0. Chuẩn bị

Chạy PostgreSQL và cài dependency nếu cần:

```bash
docker compose up -d postgres
uv pip install -r pipeline/requirements.txt
```

Mở notebook bằng kernel `.venv`. Nếu database không chạy ở localhost, đặt `EDA_DATABASE_URL`. Notebook không hiển thị connection string để tránh lộ mật khẩu.

In [65]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

ROOT = Path.cwd().resolve()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

# .env của Docker thường dùng hostname `postgres`; notebook trên host dùng localhost.
# EDA_DATABASE_URL sẽ override toàn bộ nếu được thiết lập.
DATABASE_URL = os.getenv("EDA_DATABASE_URL") or (
    f"postgresql://{os.getenv('POSTGRES_USER', 'crypto')}"
    f":{os.getenv('POSTGRES_PASSWORD', 'crypto')}"
    f"@localhost:{os.getenv('POSTGRES_PORT', '5432')}"
    f"/{os.getenv('POSTGRES_DB', 'crypto_dw')}"
)
print(f"Repository: {ROOT}")
print("Database: localhost hoặc EDA_DATABASE_URL")

Repository: /home/duong/CoinSight
Database: localhost hoặc EDA_DATABASE_URL


## 1. Cấu hình

- `SELECTED_SYMBOLS = None` lấy **toàn bộ coin có dữ liệu** trong warehouse. Chỉ truyền list khi muốn giảm phạm vi.
- `INTERVAL = "1d"` phù hợp với full history; `"1h"` chi tiết hơn nhưng toàn bộ coin từ 2020 có thể khá nặng.
- `END_DATE = None` nghĩa là lấy đến candle mới nhất. End date là mốc exclusive.

In [66]:
SOURCE_CODE = "binance"
START_DATE = "2016-01-01"
END_DATE = None
INTERVAL = "1d"                 # "1d" hoặc "1h"
SELECTED_SYMBOLS =  ["BTC", "ETH", "BNB", "ADA", "DOGE", "TRX", "LINK", "LTC", "BCH", "ATOM", "ETC"]            # None = tất cả coin trong DW; hoặc ["BTC", "ETH", ...]

DEFAULT_CANDLE_SYMBOL = "BTC"
DEFAULT_MA_WINDOW = 30             # số candle
MIN_CORRELATION_OBSERVATIONS = 30
DEFAULT_ROLLING_CORR_WINDOW = 30   # số candle
USE_LOG_PRICE_SCALE = True

## 2. Đọc dữ liệu từ Data Warehouse

Daily đọc `dw.fact_ohlcv_daily`; hourly đọc `dw.fact_ohlcv_hourly`. Category luôn lấy từ `dw.dim_asset`, vì vậy nhóm mới thêm vào dataset sẽ tự xuất hiện trong EDA.

In [67]:
def query_frame(sql: str, params: tuple = ()) -> pd.DataFrame:
    with psycopg2.connect(DATABASE_URL) as connection, connection.cursor() as cursor:
        cursor.execute(sql, params)
        columns = [column.name for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)


def load_asset_catalog(source_code: str = SOURCE_CODE) -> pd.DataFrame:
    return query_frame(
        """
        select upper(a.symbol) as symbol, a.name,
               coalesce(a.category, 'unknown') as category,
               min(f.open_time) as first_hour, max(f.open_time) as latest_hour,
               count(*)::bigint as hourly_rows
        from dw.fact_ohlcv_hourly f
        join dw.dim_asset a on a.asset_key = f.asset_key
        join dw.dim_source s on s.source_key = f.source_key
        where s.source_code = %s
        group by a.symbol, a.name, a.category
        order by coalesce(a.category, 'unknown'), a.symbol
        """,
        (source_code,),
    )


def load_ohlcv(symbols, start, end=None, interval="1d", source_code=SOURCE_CODE):
    symbols = sorted({symbol.upper() for symbol in symbols})
    if not symbols:
        raise ValueError("symbols must contain at least one symbol")
    if interval not in {"1d", "1h"}:
        raise ValueError("interval must be '1d' or '1h'")

    if interval == "1d":
        sql = """
            select upper(a.symbol) as symbol, coalesce(a.category, 'unknown') as category,
                   d.full_date::timestamptz as open_time,
                   f.open_price::float8 as open_price, f.high_price::float8 as high_price,
                   f.low_price::float8 as low_price, f.close_price::float8 as close_price,
                   f.volume_base::float8 as volume_base, f.volume_quote::float8 as volume_quote,
                   f.trade_count::bigint as trade_count
            from dw.fact_ohlcv_daily f
            join dw.dim_date d on d.date_key = f.date_key
            join dw.dim_asset a on a.asset_key = f.asset_key
            join dw.dim_source s on s.source_key = f.source_key
            where s.source_code = %s and upper(a.symbol) = any(%s)
              and d.full_date >= %s::date
              and (%s::date is null or d.full_date < %s::date)
            order by a.symbol, d.full_date
        """
    else:
        sql = """
            select upper(a.symbol) as symbol, coalesce(a.category, 'unknown') as category,
                   f.open_time, f.open_price::float8 as open_price,
                   f.high_price::float8 as high_price, f.low_price::float8 as low_price,
                   f.close_price::float8 as close_price, f.volume_base::float8 as volume_base,
                   f.volume_quote::float8 as volume_quote, f.trade_count::bigint as trade_count
            from dw.fact_ohlcv_hourly f
            join dw.dim_asset a on a.asset_key = f.asset_key
            join dw.dim_source s on s.source_key = f.source_key
            where s.source_code = %s and upper(a.symbol) = any(%s)
              and f.open_time >= %s::timestamptz
              and (%s::timestamptz is null or f.open_time < %s::timestamptz)
            order by a.symbol, f.open_time
        """

    frame = query_frame(sql, (source_code, symbols, start, end, end))
    if not frame.empty:
        frame["open_time"] = pd.to_datetime(frame["open_time"], utc=True)
        frame = frame.sort_values(["symbol", "open_time"]).reset_index(drop=True)
    return frame


def validate_ohlcv(frame):
    checked = frame.copy()
    checked["duplicate_key"] = checked.duplicated(["symbol", "open_time"], keep=False)
    checked["valid_ohlc"] = (
        checked[["open_price", "high_price", "low_price", "close_price"]].gt(0).all(axis=1)
        & checked["high_price"].ge(checked[["open_price", "close_price", "low_price"]].max(axis=1))
        & checked["low_price"].le(checked[["open_price", "close_price"]].min(axis=1))
    )
    return checked

In [68]:
try:
    asset_catalog = load_asset_catalog()
except psycopg2.OperationalError as error:
    raise RuntimeError(
        "Không kết nối được PostgreSQL. Chạy `docker compose up -d postgres` "
        "và kiểm tra EDA_DATABASE_URL nếu DB không ở localhost."
    ) from error

display(asset_catalog)

available_symbols = asset_catalog["symbol"].tolist()
if SELECTED_SYMBOLS is None:
    analysis_symbols = available_symbols
else:
    requested = list(dict.fromkeys(symbol.upper() for symbol in SELECTED_SYMBOLS))
    missing = sorted(set(requested) - set(available_symbols))
    if missing:
        print("Không có dữ liệu trong DW, bỏ qua:", missing)
    analysis_symbols = [symbol for symbol in requested if symbol in available_symbols]

if not analysis_symbols:
    raise RuntimeError("Không còn symbol nào để phân tích")

ohlcv = validate_ohlcv(load_ohlcv(analysis_symbols, START_DATE, END_DATE, INTERVAL))
if ohlcv.empty:
    raise RuntimeError("Query không trả về candle nào; kiểm tra khoảng ngày và source")

print(f"Loaded {len(ohlcv):,} {INTERVAL} candles for {ohlcv['symbol'].nunique()} symbols")
print(f"Categories ({ohlcv['category'].nunique()}): {sorted(ohlcv['category'].unique())}")
print(f"Range: {ohlcv['open_time'].min()} → {ohlcv['open_time'].max()}")

,symbol,name,category,first_hour,latest_hour,hourly_rows
0,UNI,Uniswap,defi,2020-09-17 03:00:00+00:00,2026-10-18 23:00:00+00:00,53330
1,BNB,BNB,exchange,2017-11-06 03:00:00+00:00,2026-10-18 23:00:00+00:00,78290
2,ADA,Cardano,layer-1,2018-04-17 04:00:00+00:00,2026-10-18 23:00:00+00:00,74477
3,ATOM,Cosmos,layer-1,2019-04-29 04:00:00+00:00,2026-10-18 23:00:00+00:00,65463
4,AVAX,Avalanche,layer-1,2020-09-22 06:00:00+00:00,2026-10-18 23:00:00+00:00,53207
5,BTC,Bitcoin,layer-1,2017-08-17 04:00:00+00:00,2026-10-18 23:00:00+00:00,80226
6,DOT,Polkadot,layer-1,2020-08-18 23:00:00+00:00,2026-10-18 23:00:00+00:00,54030
7,ETC,Ethereum Classic,layer-1,2018-06-12 02:00:00+00:00,2026-10-18 23:00:00+00:00,73145
8,ETH,Ethereum,layer-1,2017-08-17 04:00:00+00:00,2026-10-18 23:00:00+00:00,80226
9,NEAR,NEAR Protocol,layer-1,2020-10-14 05:00:00+00:00,2026-10-18 23:00:00+00:00,52680


Loaded 33,154 1d candles for 11 symbols
Categories (5): ['exchange', 'layer-1', 'meme', 'oracle', 'payments']
Range: 2017-08-17 00:00:00+00:00 → 2026-10-18 00:00:00+00:00


## 3. Coverage và chất lượng

`missing_candles` chỉ đo khoảng trống giữa candle đầu và cuối của coin. Coin niêm yết muộn không bị coi là thiếu trước ngày niêm yết. Bảng tổng hợp thứ hai cho biết dataset có những category nào và bao nhiêu coin mỗi nhóm.

In [69]:
STEP = pd.Timedelta(days=1) if INTERVAL == "1d" else pd.Timedelta(hours=1)

def coverage_row(group):
    first, last = group["open_time"].min(), group["open_time"].max()
    expected = int((last - first) / STEP) + 1
    return pd.Series({
        "category": group["category"].iloc[0],
        "first_candle": first, "last_candle": last,
        "rows": len(group), "expected_rows": expected,
        "missing_candles": expected - group["open_time"].nunique(),
        "duplicate_rows": int(group["duplicate_key"].sum()),
        "invalid_ohlc_rows": int((~group["valid_ohlc"]).sum()),
    })

coverage = ohlcv.groupby("symbol", sort=True).apply(coverage_row, include_groups=False)
display(coverage.sort_values(["category", "symbol"]))

category_coverage = (coverage.reset_index().groupby("category").agg(
    coin_count=("symbol", "nunique"),
    coins=("symbol", lambda values: ", ".join(sorted(values))),
    earliest_data=("first_candle", "min"),
    latest_data=("last_candle", "max"),
    total_rows=("rows", "sum"),
).sort_values(["coin_count", "category"], ascending=[False, True]))
display(category_coverage)

,category,first_candle,last_candle,rows,expected_rows,missing_candles,duplicate_rows,invalid_ohlc_rows
symbol,,,,,,,,
BNB,exchange,2017-11-06 00:00:00+00:00,2026-10-18 00:00:00+00:00,3269,3269,0,0,0
ADA,layer-1,2018-04-17 00:00:00+00:00,2026-10-18 00:00:00+00:00,3107,3107,0,0,0
ATOM,layer-1,2019-04-29 00:00:00+00:00,2026-10-18 00:00:00+00:00,2730,2730,0,0,0
BTC,layer-1,2017-08-17 00:00:00+00:00,2026-10-18 00:00:00+00:00,3350,3350,0,0,0
ETC,layer-1,2018-06-12 00:00:00+00:00,2026-10-18 00:00:00+00:00,3051,3051,0,0,0
ETH,layer-1,2017-08-17 00:00:00+00:00,2026-10-18 00:00:00+00:00,3350,3350,0,0,0
TRX,layer-1,2018-06-11 00:00:00+00:00,2026-10-18 00:00:00+00:00,3052,3052,0,0,0
DOGE,meme,2019-07-05 00:00:00+00:00,2026-10-18 00:00:00+00:00,2663,2663,0,0,0
LINK,oracle,2019-01-16 00:00:00+00:00,2026-10-18 00:00:00+00:00,2833,2833,0,0,0


,coin_count,coins,earliest_data,latest_data,total_rows
category,,,,,
layer-1,6,"ADA, ATOM, BTC, ETC, ETH, TRX",2017-08-17 00:00:00+00:00,2026-10-18 00:00:00+00:00,18640
payments,2,"BCH, LTC",2017-12-13 00:00:00+00:00,2026-10-18 00:00:00+00:00,5749
exchange,1,BNB,2017-11-06 00:00:00+00:00,2026-10-18 00:00:00+00:00,3269
meme,1,DOGE,2019-07-05 00:00:00+00:00,2026-10-18 00:00:00+00:00,2663
oracle,1,LINK,2019-01-16 00:00:00+00:00,2026-10-18 00:00:00+00:00,2833


## 4. Trend của toàn bộ coin

Mỗi coin được rebased về `100` tại candle đầu tiên của chính nó, nên đây là mức tăng/giảm tương đối. Trục log giúp so sánh coin tăng nhiều lần. Plotly hỗ trợ zoom, pan và bật/tắt từng coin trong legend.

In [70]:
close_panel = ohlcv.pivot(index="open_time", columns="symbol", values="close_price").sort_index()

def rebase_to_100(series):
    valid = series.dropna()
    return series if valid.empty else series / valid.iloc[0] * 100.0

normalized_close = close_panel.apply(rebase_to_100)
trend_long = normalized_close.rename_axis("open_time").reset_index().melt(
    id_vars="open_time", var_name="symbol", value_name="index_100"
).dropna()
trend_figure = px.line(
    trend_long, x="open_time", y="index_100", color="symbol",
    title=f"All available coins — normalized close trend ({INTERVAL})",
    labels={"open_time": "UTC time", "index_100": "Growth index (first candle = 100)"},
)
trend_figure.update_yaxes(type="log" if USE_LOG_PRICE_SCALE else "linear")
trend_figure.update_layout(height=700, hovermode="x unified", legend_title_text="Symbol")
trend_figure.show()

### Rolling regression trend: các coin đổi xu hướng cùng nhau?

Thay vì biểu diễn tăng/giảm của từng candle bằng 0/1, tại mỗi thời điểm ta fit hồi quy tuyến tính `log(close) = intercept + slope × time` trên `N` candle gần nhất. Plot đầu là mức return mà đường hồi quy ước lượng từ đầu đến cuối window: trên `0%` là xu hướng tăng, dưới `0%` là xu hướng giảm. Plot nhị phân ngay bên dưới chuyển kết quả đó thành `1 = trend > 0%` và `0 = trend < 0%`. Window lớn hơn cho đường mượt hơn nhưng phản ứng chậm hơn.

> Đây là rolling regression mô tả trend, không phải dự báo tương lai. Giữ `Ctrl`/`Cmd` để chọn nhiều coin; chọn quá nhiều coin hoặc toàn bộ hourly history sẽ làm biểu đồ nặng.

In [71]:
from plotly.subplots import make_subplots

_regression_trend_cache = {}

def rolling_regression_trend(series, window):
    cache_key = (series.name, int(window))
    if cache_key in _regression_trend_cache:
        return _regression_trend_cache[cache_key]

    log_price = np.log(series.where(series > 0))
    x = np.arange(window, dtype=float)
    centered_x = x - x.mean()
    denominator = float(np.dot(centered_x, centered_x))

    # Vì sum(centered_x) = 0, dot(centered_x, y) / denominator là OLS slope.
    slope = log_price.rolling(window, min_periods=window).apply(
        lambda values: float(np.dot(centered_x, values) / denominator),
        raw=True,
    )
    # Quy đổi slope log-price thành fitted return từ đầu tới cuối window.
    fitted_window_return = np.expm1(slope * (window - 1)) * 100.0
    _regression_trend_cache[cache_key] = fitted_window_return
    return fitted_window_return

def make_regression_trends(symbols, window):
    symbols = list(symbols)
    if not symbols:
        return go.Figure().update_layout(title="Hãy chọn ít nhất một coin")

    figure = make_subplots(
        rows=len(symbols), cols=1, shared_xaxes=True,
        vertical_spacing=min(0.04, 0.3 / len(symbols)),
        subplot_titles=symbols,
    )
    for row, symbol in enumerate(symbols, start=1):
        trend = rolling_regression_trend(close_panel[symbol], window)
        figure.add_trace(
            go.Scatter(
                x=trend.index, y=trend, mode="lines",
                line={"width": 1.5}, name=symbol,
                showlegend=False, connectgaps=False,
                hovertemplate=(
                    f"{symbol}<br>%{{x}}<br>fitted {window}-candle return=%{{y:.2f}}%"
                    "<extra></extra>"
                ),
            ),
            row=row, col=1,
        )
        figure.add_hline(y=0, line_dash="dash", line_color="gray", row=row, col=1)
        figure.update_yaxes(title_text="Trend %", zeroline=True, row=row, col=1)

    figure.update_layout(
        title=f"Rolling log-price regression trend — window={window} {INTERVAL} candles",
        height=max(450, 180 * len(symbols)),
        hovermode="x unified",
        margin={"t": 90, "b": 50},
    )
    figure.update_xaxes(rangeslider_visible=False, row=len(symbols), col=1)
    return figure

def make_regression_direction_states(symbols, window):
    symbols = list(symbols)
    if not symbols:
        return go.Figure().update_layout(title="Hãy chọn ít nhất một coin")

    figure = make_subplots(
        rows=len(symbols), cols=1, shared_xaxes=True,
        vertical_spacing=min(0.04, 0.3 / len(symbols)),
        subplot_titles=symbols,
    )
    for row, symbol in enumerate(symbols, start=1):
        trend = rolling_regression_trend(close_panel[symbol], window)
        state = pd.Series(
            np.where(trend.gt(0), 1.0, np.where(trend.lt(0), 0.0, np.nan)),
            index=trend.index, name=symbol,
        )
        figure.add_trace(
            go.Scatter(
                x=state.index, y=state, mode="lines",
                line={"shape": "hv", "width": 1.5},
                name=symbol, showlegend=False, connectgaps=False,
                customdata=trend,
                hovertemplate=(
                    f"{symbol}<br>%{{x}}<br>state=%{{y:.0f}}"
                    "<br>regression trend=%{customdata:.2f}%<extra></extra>"
                ),
            ),
            row=row, col=1,
        )
        figure.update_yaxes(
            range=[-0.15, 1.15], tickmode="array",
            tickvals=[0, 1], ticktext=["0 ↓", "1 ↑"],
            fixedrange=True, row=row, col=1,
        )

    figure.update_layout(
        title=f"Binary rolling-regression state — window={window} {INTERVAL} candles",
        height=max(420, 145 * len(symbols)),
        hovermode="x unified",
        margin={"t": 90, "b": 50},
    )
    figure.update_xaxes(rangeslider_visible=False, row=len(symbols), col=1)
    return figure

regression_defaults = tuple(
    symbol for symbol in ["BTC", "ETH", "BNB", "SOL", "XRP"]
    if symbol in close_panel.columns
)
if not regression_defaults:
    regression_defaults = tuple(close_panel.columns[: min(5, len(close_panel.columns))])

regression_symbol_control = widgets.SelectMultiple(
    options=sorted(close_panel.columns), value=regression_defaults,
    description="Coins:", rows=min(10, len(close_panel.columns)),
    layout=widgets.Layout(width="260px"),
)
default_regression_window = 30 if INTERVAL == "1d" else 168
max_regression_window = min(720, max(default_regression_window, len(close_panel) // 2))
regression_window_control = widgets.IntSlider(
    value=default_regression_window, min=7, max=max_regression_window,
    step=1 if INTERVAL == "1d" else 24, description="Window:",
    continuous_update=False, layout=widgets.Layout(width="450px"),
)

def render_regression_trends(symbols, window):
    display(make_regression_trends(symbols, window))
    display(make_regression_direction_states(symbols, window))

regression_output = widgets.interactive_output(
    render_regression_trends,
    {"symbols": regression_symbol_control, "window": regression_window_control},
)
display(
    widgets.VBox([regression_window_control, regression_symbol_control]),
    regression_output,
)

Output()

## 5. Candlestick và moving average tương tác

Chọn bất kỳ coin nào trong dataset và số candle của moving average. Có thể zoom bằng kéo chuột, scroll, range selector, range slider hoặc toolbar Plotly.

In [72]:
def make_candlestick(symbol, ma_window):
    coin = ohlcv.loc[ohlcv["symbol"] == symbol].copy()
    coin["moving_average"] = coin["close_price"].rolling(ma_window, min_periods=ma_window).mean()
    figure = go.Figure()
    figure.add_trace(go.Candlestick(
        x=coin["open_time"], open=coin["open_price"], high=coin["high_price"],
        low=coin["low_price"], close=coin["close_price"], name=f"{symbol} OHLC",
    ))
    figure.add_trace(go.Scatter(
        x=coin["open_time"], y=coin["moving_average"], mode="lines",
        name=f"MA({ma_window})", line={"color": "#f0a202", "width": 1.5},
    ))
    figure.update_layout(
        title=f"{symbol}/USDT candlestick — {INTERVAL}", height=700,
        xaxis_title="UTC time", yaxis_title="Price (USDT)", hovermode="x unified",
        xaxis={"rangeslider": {"visible": True}, "rangeselector": {"buttons": [
            {"count": 7, "label": "7d", "step": "day", "stepmode": "backward"},
            {"count": 1, "label": "1m", "step": "month", "stepmode": "backward"},
            {"count": 3, "label": "3m", "step": "month", "stepmode": "backward"},
            {"count": 1, "label": "1y", "step": "year", "stepmode": "backward"},
            {"step": "all", "label": "All"},
        ]}},
    )
    return figure

loaded_symbols = sorted(ohlcv["symbol"].unique())
default_symbol = DEFAULT_CANDLE_SYMBOL if DEFAULT_CANDLE_SYMBOL in loaded_symbols else loaded_symbols[0]
symbol_control = widgets.Dropdown(options=loaded_symbols, value=default_symbol, description="Coin:")
ma_control = widgets.IntSlider(
    value=DEFAULT_MA_WINDOW, min=2, max=200, description="MA window:", continuous_update=False
)
def render_candlestick(symbol, ma_window):
    display(make_candlestick(symbol, ma_window))

candle_output = widgets.interactive_output(
    render_candlestick, {"symbol": symbol_control, "ma_window": ma_control}
)
display(widgets.HBox([symbol_control, ma_control]), candle_output)

Output()

## 6. Return, volatility và drawdown

Log-return một kỳ là `ln(close[t] / close[t-1])`. Volatility được năm hóa theo 365 ngày hoặc 365×24 giờ vì crypto giao dịch liên tục.

In [73]:
log_returns = np.log(close_panel / close_panel.shift(1))
periods_per_year = 365 if INTERVAL == "1d" else 365 * 24
category_by_symbol = (ohlcv[["symbol", "category"]].drop_duplicates("symbol")
                      .set_index("symbol")["category"].to_dict())

summary_rows = []
for symbol in close_panel.columns:
    prices, returns = close_panel[symbol].dropna(), log_returns[symbol].dropna()
    if prices.empty or returns.empty:
        continue
    drawdown = prices / prices.cummax() - 1.0
    summary_rows.append({
        "symbol": symbol, "category": category_by_symbol[symbol],
        "observations": len(prices), "total_return": prices.iloc[-1] / prices.iloc[0] - 1,
        "mean_abs_log_return": returns.abs().mean(),
        "annualized_volatility": returns.std() * np.sqrt(periods_per_year),
        "worst_period_return": returns.min(), "best_period_return": returns.max(),
        "max_drawdown": drawdown.min(),
    })
return_summary = pd.DataFrame(summary_rows).set_index("symbol")
display(return_summary.sort_values("annualized_volatility", ascending=False))

,category,observations,total_return,mean_abs_log_return,annualized_volatility,worst_period_return,best_period_return,max_drawdown
symbol,,,,,,,,
DOGE,meme,2663,19.526601,0.035297,1.251446,-0.485146,1.594144,-0.923313
LINK,oracle,2833,23.598570,0.038894,1.066631,-0.649747,0.489638,-0.901796
ATOM,layer-1,2730,-0.552025,0.038105,1.060670,-0.617275,0.286644,-0.972351
ADA,layer-1,3107,-0.088211,0.035758,0.987695,-0.533075,0.543209,-0.951517
ETC,layer-1,3051,-0.470201,0.033236,0.982150,-0.573024,0.357542,-0.954581
BCH,payments,2517,0.270411,0.032175,0.960053,-0.616458,0.460115,-0.942488
BNB,exchange,3269,459.445576,0.029849,0.933108,-0.582287,0.532404,-0.800177
LTC,payments,3232,-0.796145,0.032339,0.912726,-0.486668,0.284906,-0.934432
ETH,layer-1,3350,6.981159,0.030624,0.871051,-0.590534,0.233750,-0.939655


## 7. Heatmap giữa toàn bộ coin

Correlation được tính pairwise tại timestamp cả hai coin đều có return. Coin niêm yết muộn có ít overlap hơn, nên xem cùng ma trận số quan sát. Symbol được sắp theo category để các cụm nhóm nằm cạnh nhau.

In [74]:
ordered_symbols = sorted(log_returns.columns, key=lambda s: (category_by_symbol.get(s, "unknown"), s))

def correlation_figure(symbols, title):
    correlation = log_returns[list(symbols)].corr(
        method="pearson", min_periods=MIN_CORRELATION_OBSERVATIONS
    )
    figure = go.Figure(go.Heatmap(
        z=correlation.to_numpy(), x=correlation.columns, y=correlation.index,
        zmin=-1, zmax=1, zmid=0, colorscale="RdBu_r",
        text=correlation.round(2).to_numpy(), texttemplate="%{text}",
        colorbar={"title": "Pearson r"}, hoverongaps=False,
    ))
    figure.update_layout(title=title, height=max(550, 35 * len(symbols)))
    return figure

display(correlation_figure(ordered_symbols, f"All coins — {INTERVAL} log-return correlation"))
valid = log_returns[ordered_symbols].notna().astype(int)
overlap_counts = valid.T.dot(valid)
display(overlap_counts.style.background_gradient(cmap="Blues").set_caption(
    "Pairwise overlapping return observations"
))

symbol,BNB,ADA,ATOM,BTC,ETC,ETH,TRX,DOGE,LINK,BCH,LTC
symbol,,,,,,,,,,,
BNB,3268,3106,2729,3268,3050,3268,3051,2662,2832,2516,3231
ADA,3106,3106,2729,3106,3050,3106,3051,2662,2832,2516,3106
ATOM,2729,2729,2729,2729,2729,2729,2729,2662,2729,2516,2729
BTC,3268,3106,2729,3349,3050,3349,3051,2662,2832,2516,3231
ETC,3050,3050,2729,3050,3050,3050,3050,2662,2832,2516,3050
ETH,3268,3106,2729,3349,3050,3349,3051,2662,2832,2516,3231
TRX,3051,3051,2729,3051,3050,3051,3051,2662,2832,2516,3051
DOGE,2662,2662,2662,2662,2662,2662,2662,2662,2662,2516,2662
LINK,2832,2832,2729,2832,2832,2832,2832,2662,2832,2516,2832


### 7.1. Lead–lag correlation: coin khác tại `t` với target tại `t+1`

Cell này đo `corr(source_return[t], target_return[t+1])`. Target return tại `t+1` là `log(close_target[t+1] / close_target[t])`; source return tại `t` chỉ dùng giá đã biết đến `t`. Vì vậy đây là phép kiểm tra lead–lag đúng thời điểm, khác với correlation đồng thời `corr(source[t], target[t])`.

> Correlation khác 0 chưa chứng minh quan hệ nhân quả hoặc đảm bảo model tốt hơn. Cần xác nhận bằng validation/walk-forward và xem tính ổn định theo từng giai đoạn.

In [75]:
def lead_lag_correlations(target_symbol):
    # Dời return của target tại t+1 về index t để ghép với source return tại t.
    target_next_return = log_returns[target_symbol].copy()
    target_next_return.index = target_next_return.index - STEP
    target_next_return = target_next_return.reindex(log_returns.index)

    rows = []
    for source_symbol in log_returns.columns:
        if source_symbol == target_symbol:
            continue
        aligned = pd.concat(
            [
                log_returns[source_symbol].rename("source_return_t"),
                target_next_return.rename("target_return_t_plus_1"),
            ],
            axis=1,
        ).dropna()
        if len(aligned) < MIN_CORRELATION_OBSERVATIONS:
            pearson = np.nan
            spearman = np.nan
        else:
            pearson = aligned["source_return_t"].corr(
                aligned["target_return_t_plus_1"], method="pearson"
            )
            spearman = aligned["source_return_t"].corr(
                aligned["target_return_t_plus_1"], method="spearman"
            )
        rows.append({
            "source_symbol": source_symbol,
            "category": category_by_symbol.get(source_symbol, "unknown"),
            "pearson": pearson,
            "spearman": spearman,
            "observations": len(aligned),
        })
    return pd.DataFrame(rows)

def make_lead_lag_figure(target_symbol):
    correlations = lead_lag_correlations(target_symbol).dropna(subset=["pearson"])
    correlations = correlations.sort_values("pearson").reset_index(drop=True)
    if correlations.empty:
        return go.Figure().update_layout(title="Không đủ dữ liệu để tính correlation"), correlations

    figure = px.bar(
        correlations, x="source_symbol", y="pearson", color="category",
        text="pearson",
        custom_data=["spearman", "observations"],
        title=(
            f"Lead–lag correlation: source return[t] → "
            f"{target_symbol} return[t+1] ({INTERVAL})"
        ),
        labels={"source_symbol": "Source coin at t", "pearson": "Pearson correlation"},
    )
    figure.update_traces(
        texttemplate="%{text:.3f}", textposition="outside",
        hovertemplate=(
            "source=%{x}<br>Pearson=%{y:.4f}"
            "<br>Spearman=%{customdata[0]:.4f}"
            "<br>observations=%{customdata[1]:,.0f}<extra></extra>"
        ),
    )
    limit = max(0.05, float(correlations["pearson"].abs().max()) * 1.2)
    figure.update_yaxes(range=[-limit, limit], zeroline=True, zerolinewidth=2)
    figure.add_hline(y=0, line_color="gray")
    figure.update_layout(height=550, hovermode="x")
    return figure, correlations

lead_lag_target_default = "BTC" if "BTC" in loaded_symbols else loaded_symbols[0]
lead_lag_target_control = widgets.Dropdown(
    options=loaded_symbols, value=lead_lag_target_default, description="Target:"
)

def render_lead_lag(target_symbol):
    figure, correlations = make_lead_lag_figure(target_symbol)
    display(figure)
    ranked = correlations.assign(abs_pearson=correlations["pearson"].abs()).sort_values(
        "abs_pearson", ascending=False
    )
    display(ranked[["source_symbol", "category", "pearson", "spearman", "observations"]])

lead_lag_output = widgets.interactive_output(
    render_lead_lag, {"target_symbol": lead_lag_target_control}
)
display(lead_lag_target_control, lead_lag_output)

Dropdown(description='Target:', index=4, options=('ADA', 'ATOM', 'BCH', 'BNB', 'BTC', 'DOGE', 'ETC', 'ETH', 'L…

Output()

### 7.2. Lead–lag correlation có ổn định qua từng năm không?

Heatmap tiếp tục đo `corr(source_return[t], target_return[t+1])`, nhưng tính riêng cho từng năm theo timestamp `t`. Một feature đáng nghiên cứu hơn khi correlation giữ cùng dấu qua nhiều năm; correlation toàn kỳ có thể che giấu việc tín hiệu đổi dấu giữa các market regime. Ô trống là năm chưa đủ số quan sát. Năm đầu/cuối dataset có thể là năm không đầy đủ.

> `Dominant sign share = 1.0` nghĩa là tất cả các năm hợp lệ cùng dấu, không có nghĩa correlation mạnh hoặc có ý nghĩa kinh tế.

In [76]:
MIN_YEARLY_OBSERVATIONS = 30

def yearly_lead_lag_correlations(target_symbol):
    target_next_return = log_returns[target_symbol].copy()
    target_next_return.index = target_next_return.index - STEP
    target_next_return = target_next_return.reindex(log_returns.index)

    rows = []
    for source_symbol in log_returns.columns:
        if source_symbol == target_symbol:
            continue
        aligned = pd.concat(
            [
                log_returns[source_symbol].rename("source_return_t"),
                target_next_return.rename("target_return_t_plus_1"),
            ],
            axis=1,
        ).dropna()
        aligned["year"] = aligned.index.year
        for year, period in aligned.groupby("year"):
            if len(period) < MIN_YEARLY_OBSERVATIONS:
                pearson = np.nan
                spearman = np.nan
            else:
                pearson = period["source_return_t"].corr(
                    period["target_return_t_plus_1"], method="pearson"
                )
                spearman = period["source_return_t"].corr(
                    period["target_return_t_plus_1"], method="spearman"
                )
            rows.append({
                "source_symbol": source_symbol,
                "category": category_by_symbol.get(source_symbol, "unknown"),
                "year": int(year),
                "pearson": pearson,
                "spearman": spearman,
                "observations": len(period),
            })
    return pd.DataFrame(rows)

def correlation_stability_summary(yearly, method):
    rows = []
    for source_symbol, group in yearly.groupby("source_symbol"):
        values = group[method].dropna()
        if values.empty:
            continue
        positive_years = int(values.gt(0).sum())
        negative_years = int(values.lt(0).sum())
        nonzero_years = positive_years + negative_years
        dominant_share = (
            max(positive_years, negative_years) / nonzero_years
            if nonzero_years else np.nan
        )
        rows.append({
            "source_symbol": source_symbol,
            "category": category_by_symbol.get(source_symbol, "unknown"),
            "mean_correlation": values.mean(),
            "std_between_years": values.std(ddof=0),
            "positive_years": positive_years,
            "negative_years": negative_years,
            "valid_years": len(values),
            "dominant_sign_share": dominant_share,
        })
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary["abs_mean_correlation"] = summary["mean_correlation"].abs()
        summary = summary.sort_values(
            ["dominant_sign_share", "abs_mean_correlation"],
            ascending=[False, False],
        )
    return summary

def make_yearly_lead_lag_figure(target_symbol, method):
    yearly = yearly_lead_lag_correlations(target_symbol)
    values = yearly.pivot(index="source_symbol", columns="year", values=method)
    counts = yearly.pivot(index="source_symbol", columns="year", values="observations")
    symbol_order = sorted(values.index, key=lambda s: (category_by_symbol.get(s, "unknown"), s))
    values = values.reindex(symbol_order)
    counts = counts.reindex(index=symbol_order, columns=values.columns)

    finite_values = values.to_numpy(dtype=float)
    finite_values = np.abs(finite_values[np.isfinite(finite_values)])
    color_limit = max(0.10, float(finite_values.max())) if len(finite_values) else 0.10
    labels = values.apply(lambda column: column.map(
        lambda value: "" if pd.isna(value) else f"{value:.2f}"
    ))

    figure = go.Figure(go.Heatmap(
        z=values.to_numpy(), x=values.columns, y=values.index,
        zmin=-color_limit, zmax=color_limit, zmid=0, colorscale="RdBu_r",
        text=labels.to_numpy(), texttemplate="%{text}",
        customdata=counts.to_numpy(), hoverongaps=False,
        hovertemplate=(
            "source=%{y}<br>year=%{x}<br>correlation=%{z:.4f}"
            "<br>observations=%{customdata:,.0f}<extra></extra>"
        ),
        colorbar={"title": method.title()},
    ))
    figure.update_layout(
        title=(
            f"Yearly {method.title()} stability: source return[t] → "
            f"{target_symbol} return[t+1]"
        ),
        height=max(550, 35 * len(values.index)),
        xaxis_title="Year of source timestamp t", yaxis_title="Source coin",
    )
    return figure, correlation_stability_summary(yearly, method)

stability_target_control = widgets.Dropdown(
    options=loaded_symbols, value=lead_lag_target_default, description="Target:"
)
stability_method_control = widgets.Dropdown(
    options=[("Spearman", "spearman"), ("Pearson", "pearson")],
    value="spearman", description="Method:"
)

def render_yearly_lead_lag(target_symbol, method):
    figure, summary = make_yearly_lead_lag_figure(target_symbol, method)
    display(figure)
    display(summary.drop(columns=["abs_mean_correlation"], errors="ignore"))

yearly_lead_lag_output = widgets.interactive_output(
    render_yearly_lead_lag,
    {"target_symbol": stability_target_control, "method": stability_method_control},
)
display(widgets.HBox([stability_target_control, stability_method_control]), yearly_lead_lag_output)

Output()

## 8. Heatmap của từng nhóm coin

Dropdown tự sinh từ **tất cả category trong warehouse** có ít nhất hai coin. Ví dụ chọn `meme` để xem DOGE–SHIB; nhóm mới sẽ tự xuất hiện sau khi cập nhật `dw.dim_asset.category`.

In [77]:
categories = sorted(set(category_by_symbol.values()))
group_options = ["all"] + [
    category for category in categories
    if sum(category_by_symbol[symbol] == category for symbol in ordered_symbols) >= 2
]
default_group = "meme" if "meme" in group_options else "all"
group_control = widgets.Dropdown(options=group_options, value=default_group, description="Category:")

def render_group_correlation(category):
    symbols = ordered_symbols if category == "all" else [
        symbol for symbol in ordered_symbols if category_by_symbol[symbol] == category
    ]
    display(correlation_figure(symbols, f"{category} — {INTERVAL} log-return correlation"))

group_output = widgets.interactive_output(render_group_correlation, {"category": group_control})
display(group_control, group_output)

Dropdown(description='Category:', options=('all', 'layer-1', 'payments'), value='all')

Output()

## 9. Heatmap giữa toàn bộ nhóm coin

Mỗi category được biểu diễn bằng return trung bình bằng trọng số của các coin đang có dữ liệu ở timestamp đó. Đây là exploratory equal-weight index, chưa phải market-cap-weighted index. Nhóm chỉ có một coin vẫn xuất hiện nhưng đại diện đúng một coin.

In [78]:
category_returns = log_returns.rename(columns=category_by_symbol).T.groupby(level=0).mean().T
category_correlation = category_returns.corr(min_periods=MIN_CORRELATION_OBSERVATIONS)
category_figure = go.Figure(go.Heatmap(
    z=category_correlation.to_numpy(), x=category_correlation.columns, y=category_correlation.index,
    zmin=-1, zmax=1, zmid=0, colorscale="RdBu_r",
    text=category_correlation.round(2).to_numpy(), texttemplate="%{text}",
    colorbar={"title": "Pearson r"}, hoverongaps=False,
))
category_figure.update_layout(title=f"All asset categories — {INTERVAL} return correlation", height=600)
category_figure.show()

## 10. Rolling correlation của một cặp coin

Correlation toàn lịch sử có thể che mất thay đổi market regime. Chọn BTC–ETH hoặc bất kỳ cặp coin nào để xem correlation theo rolling window.

In [79]:
pair_a_default = "BTC" if "BTC" in loaded_symbols else loaded_symbols[0]
pair_b_default = "ETH" if "ETH" in loaded_symbols else loaded_symbols[min(1, len(loaded_symbols) - 1)]
pair_a_control = widgets.Dropdown(options=loaded_symbols, value=pair_a_default, description="Coin A:")
pair_b_control = widgets.Dropdown(options=loaded_symbols, value=pair_b_default, description="Coin B:")
rolling_control = widgets.IntSlider(
    value=DEFAULT_ROLLING_CORR_WINDOW, min=10, max=365, step=5,
    description="Window:", continuous_update=False,
)

def render_rolling_correlation(symbol_a, symbol_b, window):
    if symbol_a == symbol_b:
        print("Hãy chọn hai coin khác nhau.")
        return
    rolling = log_returns[symbol_a].rolling(
        window, min_periods=max(10, window // 2)
    ).corr(log_returns[symbol_b])
    figure = px.line(
        x=rolling.index, y=rolling.values,
        title=f"Rolling correlation: {symbol_a} vs {symbol_b} ({window} {INTERVAL} candles)",
        labels={"x": "UTC time", "y": "Pearson correlation"},
    )
    figure.add_hline(y=0, line_dash="dash", line_color="gray")
    figure.update_yaxes(range=[-1, 1])
    figure.update_layout(height=500, showlegend=False)
    display(figure)

pair_output = widgets.interactive_output(render_rolling_correlation, {
    "symbol_a": pair_a_control, "symbol_b": pair_b_control, "window": rolling_control
})
display(widgets.HBox([pair_a_control, pair_b_control, rolling_control]), pair_output)

Output()

## 11. Những candle biến động mạnh nhất

Bảng hỗ trợ phân biệt biến động thật với lỗi dữ liệu. Không tự động xóa COVID/FTX/Terra chỉ vì return lớn; cần đối chiếu OHLC, volume, timestamp và chuyển động của coin khác.

In [80]:
returns_long = log_returns.stack(future_stack=True).rename("log_return").reset_index()
returns_long["abs_log_return"] = returns_long["log_return"].abs()
returns_long["category"] = returns_long["symbol"].map(category_by_symbol)
extreme_returns = returns_long.nlargest(30, "abs_log_return")[
    ["open_time", "symbol", "category", "log_return", "abs_log_return"]
]
display(extreme_returns)

,open_time,symbol,category,log_return,abs_log_return
13865,2021-01-28 00:00:00+00:00,DOGE,meme,1.594144,1.594144
14723,2021-04-16 00:00:00+00:00,DOGE,meme,0.695129,0.695129
29325,2024-12-03 00:00:00+00:00,TRX,layer-1,0.673498,0.673498
10326,2020-03-12 00:00:00+00:00,LINK,oracle,-0.649747,0.649747
10319,2020-03-12 00:00:00+00:00,ATOM,layer-1,-0.617275,0.617275
13579,2021-01-02 00:00:00+00:00,DOGE,meme,0.616935,0.616935
10320,2020-03-12 00:00:00+00:00,BCH,payments,-0.616458,0.616458
10325,2020-03-12 00:00:00+00:00,ETH,layer-1,-0.590534,0.590534
10321,2020-03-12 00:00:00+00:00,BNB,exchange,-0.582287,0.582287
10324,2020-03-12 00:00:00+00:00,ETC,layer-1,-0.573024,0.573024


## Cách dùng kết quả trước khi làm feature v2

- Correlation thay đổi mạnh theo thời gian → cân nhắc rolling market-regime/volatility feature.
- Coin cùng category tương quan cao → global model có thể hữu ích, nhưng cần nhận diện symbol/category.
- Category chỉ có một coin hoặc overlap ngắn → chưa đủ để kết luận cho cả nhóm.
- Feature mới phải chỉ dùng dữ liệu có trước cutoff, không dùng tương lai.
- So sánh feature v1/v2 bằng cùng split, model config và seed; quyết định trên validation/walk-forward, không tune trên test.